# ETL — VISp Excitatory Whole Neuron Morphology: DataSet & DataItem

Writes one `DataSet` record (`dataset_id = "visp_exc_wnm"`, `project_id = "visp_wnm"`), one `DataItem` per cell from `FullMorphMetaData_Master.csv` (cell id = SWC filename with `.swc` stripped), and the corresponding `DataItemDataSetAssociation` links. No prerequisites; features and cluster mappings are written in `_02` and `_03`.

In [1]:
import pandas as pd
import polars as pl
import pyarrow as pa

from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Modality,
)
from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models


In [2]:
INPUT_CSV   = "/data/visp-features-and-mapping/FullMorphMetaData_Master.csv" # same as /data/exc_vis_manuscript_wnm_axon_projection/FullMorphMetaData_Master.csv
OUTPUT_ROOT = get_settings().output_root
PROJECT_ID  = "visp_wnm"
DATASET_ID  = "visp_exc_wnm"

print(f"INPUT_CSV   : {INPUT_CSV}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"PROJECT_ID  : {PROJECT_ID}")
print(f"DATASET_ID  : {DATASET_ID}")

INPUT_CSV   : /data/visp-features-and-mapping/FullMorphMetaData_Master.csv
OUTPUT_ROOT : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID  : visp_wnm
DATASET_ID  : visp_exc_wnm


## Load input CSV

In [3]:
df = pd.read_csv(INPUT_CSV, index_col=0)
# Strip the trailing .swc extension to form the cell id; otherwise values are unchanged.
df.index = df.index.str.removesuffix(".swc")
print("Shape:", df.shape)
df.head(3)

Shape: (341, 16)


,predicted_met_type,probability,ccf_soma_location,ccf_soma_location_nolayer,ccf_soma_x,ccf_soma_y,ccf_soma_z,distance_soma_moved_out_of_brain_correction,cre_line,azimuth,altitude,auto_projection_subclass,dend_derived_predicted_subclass,dend_derived_predicted_probability,local_axon_derived_subclass,met_classifier_routing_call
182709_6984-X2452-Y12423_reg,L5 ET-2,0.988,VISpm5,VISpm,8899.823,643.262,4324.473,37.416574,Ai82;Ai139_375886-182709,35.212157,3.519493,ET,ET,0.760219,NaN,ET
182709_7126-X2913-Y10535_reg,L5 ET-3,0.918,VISp5,VISp,9117.453,1064.075,3550.508,24.494897,Ai82;Ai139_375886-182709,48.447344,-0.296991,ET,ET,0.949430,NaN,ET
182724_5937-X3804-Y11955_reg,L5 ET-2,0.724,VISa5,VISa,7168.289,953.689,3959.651,42.426407,Fezf2-CreER;Ai166_405426-182724,41.861469,1.670020,ET,ET,0.928120,NaN,ET


## Write `DataSet`

In [4]:
dataset = DataSet(
    id=DATASET_ID,
    name="VISp excitatory whole neuron morphology data set",
    publication="doi.org/10.1101/2023.11.25.568393",
    modality=Modality.MORPHOLOGY.value,
    project_id=PROJECT_ID,
)
result = write_models([dataset], output_root=OUTPUT_ROOT)
print(f"DataSet written: {result.rows_written} rows")

DataSet written: 1 rows


In [5]:
# Verification
ds_verify = (
    pl.read_delta(OUTPUT_ROOT / "dataset")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == DATASET_ID))
    .filter(pl.col("id") == DATASET_ID)
)
print(ds_verify.shape)
print(ds_verify.head())
assert ds_verify.shape[0] == 1, f"Expected 1 DataSet row, got {ds_verify.shape[0]}"
assert ds_verify["id"][0] == DATASET_ID, "DataSet id mismatch"

(1, 5)
shape: (1, 5)
┌──────────────┬───────────────────────┬─────────────────────────────────┬────────────┬────────────┐
│ id           ┆ name                  ┆ publication                     ┆ modality   ┆ project_id │
│ ---          ┆ ---                   ┆ ---                             ┆ ---        ┆ ---        │
│ str          ┆ str                   ┆ str                             ┆ str        ┆ str        │
╞══════════════╪═══════════════════════╪═════════════════════════════════╪════════════╪════════════╡
│ visp_exc_wnm ┆ VISp excitatory whole ┆ doi.org/10.1101/2023.11.25.568… ┆ MORPHOLOGY ┆ visp_wnm   │
│              ┆ neuron m…             ┆                                 ┆            ┆            │
└──────────────┴───────────────────────┴─────────────────────────────────┴────────────┴────────────┘


## Write `DataItem`

In [6]:
cell_ids = df.index.tolist()

dataitems = [
    DataItem(id=cid, name=cid, project_id=PROJECT_ID)
    for cid in cell_ids
]
n_appended = write_models(dataitems, output_root=OUTPUT_ROOT).rows_written
print(f"DataItem rows appended: {n_appended} (total in batch: {len(cell_ids)})")

DataItem rows appended: 341 (total in batch: 341)


In [7]:
# Verification
di_verify = (
    pl.read_delta(OUTPUT_ROOT / "dataitem")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(di_verify.shape)
print(di_verify.head())
registered_ids = set(di_verify["id"].to_list())
assert all(cid in registered_ids for cid in cell_ids), "Some cell_ids are missing from DataItem table"
assert di_verify["id"].n_unique() == di_verify.shape[0], "Duplicate DataItem ids detected"

(341, 4)
shape: (5, 4)
┌──────────────────────────────┬──────────────────────────────┬───────────────────┬────────────┐
│ id                           ┆ name                         ┆ neuroglancer_link ┆ project_id │
│ ---                          ┆ ---                          ┆ ---               ┆ ---        │
│ str                          ┆ str                          ┆ str               ┆ str        │
╞══════════════════════════════╪══════════════════════════════╪═══════════════════╪════════════╡
│ 182709_6984-X2452-Y12423_reg ┆ 182709_6984-X2452-Y12423_reg ┆ null              ┆ visp_wnm   │
│ 182709_7126-X2913-Y10535_reg ┆ 182709_7126-X2913-Y10535_reg ┆ null              ┆ visp_wnm   │
│ 182724_5937-X3804-Y11955_reg ┆ 182724_5937-X3804-Y11955_reg ┆ null              ┆ visp_wnm   │
│ 182724_6175-X3782-Y10859_reg ┆ 182724_6175-X3782-Y10859_reg ┆ null              ┆ visp_wnm   │
│ 182724_6354-X4834-Y8105_reg  ┆ 182724_6354-X4834-Y8105_reg  ┆ null              ┆ visp_wnm   │
└──────

## Write `DataItemDataSetAssociation`

In [8]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cid,
        dataset_id=DATASET_ID,
        project_id=PROJECT_ID,
    )
    for cid in cell_ids
]
result = write_models(associations, output_root=OUTPUT_ROOT)
print(f"DataItemDataSetAssociation written: {result.rows_written} rows")

DataItemDataSetAssociation written: 341 rows


In [9]:
# Verification
assoc_verify = (
    pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association")
    .filter(pl.col("project_id") == PROJECT_ID)
    .filter(pl.col("dataset_id") == DATASET_ID)
)
print(assoc_verify.shape)
print(assoc_verify.head())
assert assoc_verify.shape[0] == len(cell_ids), (
    f"Expected {len(cell_ids)} association rows, got {assoc_verify.shape[0]}"
)
assert (assoc_verify["dataset_id"] == DATASET_ID).all(), "Not all associations point to DATASET_ID"

(341, 3)
shape: (5, 3)
┌──────────────────────────────┬──────────────┬────────────┐
│ dataitem_id                  ┆ dataset_id   ┆ project_id │
│ ---                          ┆ ---          ┆ ---        │
│ str                          ┆ str          ┆ str        │
╞══════════════════════════════╪══════════════╪════════════╡
│ 182709_6984-X2452-Y12423_reg ┆ visp_exc_wnm ┆ visp_wnm   │
│ 182709_7126-X2913-Y10535_reg ┆ visp_exc_wnm ┆ visp_wnm   │
│ 182724_5937-X3804-Y11955_reg ┆ visp_exc_wnm ┆ visp_wnm   │
│ 182724_6175-X3782-Y10859_reg ┆ visp_exc_wnm ┆ visp_wnm   │
│ 182724_6354-X4834-Y8105_reg  ┆ visp_exc_wnm ┆ visp_wnm   │
└──────────────────────────────┴──────────────┴────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataset/` | `DataSet` | 1 |
| `dataitem/` | `DataItem` | 341 |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | 341 |

**Input columns intentionally not written here:**
- `predicted_met_type`, `probability` — MET-type classification; written in a later notebook as `CellToClusterMapping`.
- `ccf_soma_location`, `ccf_soma_x/y/z` and remaining morphology metadata — written in a later notebook as `SingleCellRecon` records.